## 📌 Section 1: Project Introduction

### What is This Project?

The **Context-Aware Neural Recommendation Engine** is a machine learning engineering project that builds a personalized fashion recommendation system using real-world data from H&M.

The system will eventually:
- Suggest clothing items that a customer is most likely to purchase
- Adapt recommendations based on customer behavior and purchase history
- Serve real-time predictions via a FastAPI backend, powered by TensorFlow Recommenders and ANN search

---

### Why Does Dataset Understanding Matter?

Before building any model, a machine learning engineer **must deeply understand the data**. Skipping this step leads to:
- 🚫 Incorrect preprocessing decisions
- 🚫 Feeding wrong or noisy features to the model
- 🚫 Missing important patterns in user behavior

This notebook focuses **entirely** on understanding the dataset — no modeling, no preprocessing. Just exploration.

---

### Why Are Recommendation Systems Important?

Recommendation systems are one of the most impactful applications of ML in the real world:

| Platform | Use Case |
|----------|----------|
| Netflix | Recommends movies and shows |
| Amazon | Suggests products |
| Spotify | Curates playlists |
| H&M | Personalizes fashion suggestions |

A good recommendation engine increases **user engagement**, **conversion rate**, and **revenue** by showing the right product to the right person at the right time.

---

### 🛠️ Full Tech Stack (for reference)

| Phase | Tools |
|-------|-------|
| Data Understanding | pandas, numpy, matplotlib |
| Modeling | TensorFlow Recommenders |
| Serving | FastAPI |
| Caching | Redis |
| Orchestration | Apache Airflow |
| Search | ANN (Approximate Nearest Neighbors) |

> ⚠️ **This notebook covers only the first phase: Data Understanding.**

---

## 📦 Section 2: Import Libraries

We import only what we need for data exploration. Keeping imports minimal and organized is a good professional habit.

- **pandas**: For loading and manipulating tabular data (DataFrames)
- **numpy**: For numerical operations and array handling
- **matplotlib**: For creating visualizations and charts

In [ ]:
# ──────────────────────────────────────────────
# Standard Data Science Libraries
# ──────────────────────────────────────────────

import pandas as pd           # DataFrame operations and CSV loading
import numpy as np            # Numerical operations
import matplotlib.pyplot as plt  # Plotting and visualization
import matplotlib.ticker as mticker

# ──────────────────────────────────────────────
# Notebook Display Settings
# ──────────────────────────────────────────────

# Show all columns when displaying DataFrames
pd.set_option('display.max_columns', None)

# Limit rows shown in output for readability
pd.set_option('display.max_rows', 50)

# Format floats cleanly (2 decimal places)
pd.set_option('display.float_format', '{:.2f}'.format)

# Matplotlib style for clean, professional plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("✅ Libraries imported successfully.")

---

## 📂 Section 3: Load Datasets

The H&M dataset consists of three files:

| File | Description |
|------|-------------|
| `customers.csv` | Information about each customer (age, membership status, etc.) |
| `articles.csv` | Details about each fashion article/product |
| `transactions_train.csv` | Historical purchase records (who bought what and when) |

We load them using `pandas.read_csv()`. The `transactions` file is large, so loading it may take a moment.

In [ ]:
# ──────────────────────────────────────────────
# File Paths
# ──────────────────────────────────────────────

CUSTOMERS_PATH    = '../data/raw/customers.csv'
ARTICLES_PATH     = '../data/raw/articles.csv'
TRANSACTIONS_PATH = '../data/raw/transactions_train.csv'

# ──────────────────────────────────────────────
# Load All Three Datasets
# ──────────────────────────────────────────────

print("⏳ Loading datasets... (transactions may take a moment)")

customers    = pd.read_csv(CUSTOMERS_PATH)
articles     = pd.read_csv(ARTICLES_PATH, dtype={'article_id': str})
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    dtype={'article_id': str},       # Keep article_id as string to avoid precision issues
    parse_dates=['t_dat']            # Automatically parse the date column
)

print("✅ All datasets loaded successfully!")
print(f"   • Customers    : {customers.shape[0]:,} rows × {customers.shape[1]} columns")
print(f"   • Articles     : {articles.shape[0]:,} rows × {articles.shape[1]} columns")
print(f"   • Transactions : {transactions.shape[0]:,} rows × {transactions.shape[1]} columns")

---

## 🔍 Section 4: Basic Dataset Inspection

Before anything else, we need to understand the **structure** of each dataset:
- What columns exist?
- What data types are used?
- What does the first few rows look like?

This is always the first thing you do when you encounter a new dataset.

### 4.1 — Customers Dataset

In [ ]:
# ──────────────────────────────────────────────
# Customers: First Look
# ──────────────────────────────────────────────

print("="*60)
print("CUSTOMERS DATASET")
print("="*60)

print(f"\n📐 Shape: {customers.shape[0]:,} rows × {customers.shape[1]} columns")
print(f"\n📋 Column Names:")
print(customers.columns.tolist())

In [ ]:
# View the first 5 rows — always do this to get a 'feel' for the data
print("\n👀 First 5 Rows:")
customers.head()

In [ ]:
# .info() gives us column data types and non-null counts at a glance
# This is very useful to spot potential issues (e.g., numeric column stored as string)
print("\nℹ️  Dataset Info:")
customers.info()

### 4.2 — Articles Dataset

In [ ]:
# ──────────────────────────────────────────────
# Articles: First Look
# ──────────────────────────────────────────────

print("="*60)
print("ARTICLES DATASET")
print("="*60)

print(f"\n📐 Shape: {articles.shape[0]:,} rows × {articles.shape[1]} columns")
print(f"\n📋 Column Names:")
print(articles.columns.tolist())

In [ ]:
print("\n👀 First 5 Rows:")
articles.head()

In [ ]:
print("\nℹ️  Dataset Info:")
articles.info()

### 4.3 — Transactions Dataset

In [ ]:
# ──────────────────────────────────────────────
# Transactions: First Look
# ──────────────────────────────────────────────

print("="*60)
print("TRANSACTIONS DATASET")
print("="*60)

print(f"\n📐 Shape: {transactions.shape[0]:,} rows × {transactions.shape[1]} columns")
print(f"\n📋 Column Names:")
print(transactions.columns.tolist())

In [ ]:
print("\n👀 First 5 Rows:")
transactions.head()

In [ ]:
print("\nℹ️  Dataset Info:")
transactions.info()

---

## ❓ Section 5: Missing Value Analysis

Missing values are one of the most common data quality problems in real-world datasets.

**Why do missing values matter?**
- ML models typically cannot handle `NaN` (Not a Number) values directly
- Missing values may indicate data collection issues
- We must decide later: fill them in (impute), drop them, or keep them as a separate category

We calculate missing values as both **count** and **percentage** so we can prioritize which columns need attention.

In [ ]:
# ──────────────────────────────────────────────
# Reusable Function: Missing Value Report
# ──────────────────────────────────────────────

def missing_value_report(df, dataset_name):
    """
    Generates a clean missing value summary for a DataFrame.
    Shows count and percentage of missing values per column.
    Only displays columns that have at least 1 missing value.
    """
    print(f"\n{'='*55}")
    print(f"  Missing Value Report: {dataset_name}")
    print(f"{'='*55}")

    # Count and percentage of missing values
    missing_count = df.isnull().sum()
    missing_pct   = (missing_count / len(df)) * 100

    # Combine into a readable DataFrame
    report = pd.DataFrame({
        'Missing Count': missing_count,
        'Missing (%)': missing_pct.round(2)
    })

    # Show only columns with missing values
    report = report[report['Missing Count'] > 0].sort_values('Missing (%)', ascending=False)

    if report.empty:
        print("  ✅ No missing values found!")
    else:
        print(f"  ⚠️  {len(report)} column(s) have missing values:\n")
        print(report.to_string())

    print()

In [ ]:
# Run the missing value report for each dataset
missing_value_report(customers,    "Customers")
missing_value_report(articles,     "Articles")
missing_value_report(transactions, "Transactions")

---

## 🔁 Section 6: Duplicate Value Analysis

**Duplicate rows** are rows that appear more than once in the dataset.

In recommendation systems:
- Duplicates in `customers` or `articles` can cause double-counting and corrupt feature engineering
- Duplicates in `transactions` might be legitimate (a customer genuinely bought the same item twice) OR data entry errors

We check each dataset and report the count.

In [ ]:
# ──────────────────────────────────────────────
# Duplicate Row Check
# ──────────────────────────────────────────────

def duplicate_report(df, dataset_name):
    """
    Checks and reports the number of fully duplicate rows in a DataFrame.
    """
    n_duplicates = df.duplicated().sum()
    pct = (n_duplicates / len(df)) * 100

    status = "✅ None found" if n_duplicates == 0 else f"⚠️  {n_duplicates:,} duplicates ({pct:.2f}%)"
    print(f"  {dataset_name:<20} → {status}")


print("="*55)
print("  Duplicate Row Analysis")
print("="*55)
duplicate_report(customers,    "Customers")
duplicate_report(articles,     "Articles")
duplicate_report(transactions, "Transactions")

---

## 🔗 Section 7: Understand Dataset Relationships

Understanding **how the three datasets connect** is critical before any modeling.

```
┌──────────────┐         ┌──────────────────────┐         ┌──────────────┐
│  customers   │         │    transactions       │         │   articles   │
│──────────────│         │──────────────────────│         │──────────────│
│ customer_id  │────────►│ customer_id          │◄────────│ article_id   │
│ age          │         │ article_id           │         │ product_name │
│ club_member  │         │ t_dat (date)         │         │ product_type │
│ ...          │         │ price                │         │ color        │
└──────────────┘         │ sales_channel_id     │         │ ...          │
                         └──────────────────────┘         └──────────────┘
```

- **`customer_id`** links customers → transactions (one customer, many transactions)
- **`article_id`** links articles → transactions (one article, many transactions)
- **Transactions** is the **fact table** that connects users with products over time

In [ ]:
# ──────────────────────────────────────────────
# Verify Relationship Integrity
# ──────────────────────────────────────────────

# How many unique customers are in the transactions file?
txn_customers = transactions['customer_id'].nunique()
all_customers = customers['customer_id'].nunique()

# How many unique articles are in the transactions file?
txn_articles = transactions['article_id'].nunique()
all_articles = articles['article_id'].nunique()

print("="*55)
print("  Dataset Relationship Summary")
print("="*55)
print(f"\n  👤 Total unique customers        : {all_customers:,}")
print(f"  👤 Customers with transactions   : {txn_customers:,}")
print(f"     → {txn_customers/all_customers*100:.1f}% of customers have purchase history\n")

print(f"  👗 Total unique articles         : {all_articles:,}")
print(f"  👗 Articles with transactions    : {txn_articles:,}")
print(f"     → {txn_articles/all_articles*100:.1f}% of articles have been purchased\n")

print(f"  🧾 Total transaction records     : {len(transactions):,}")
print(f"  📅 Date range                    : {transactions['t_dat'].min().date()} to {transactions['t_dat'].max().date()}")

In [ ]:
# ──────────────────────────────────────────────
# Check for Orphan Records
# (Transactions referencing IDs not in master tables)
# ──────────────────────────────────────────────

# Customers in transactions but NOT in customers table
orphan_customers = set(transactions['customer_id']) - set(customers['customer_id'])

# Articles in transactions but NOT in articles table
orphan_articles = set(transactions['article_id']) - set(articles['article_id'])

print("\n  🔎 Orphan Record Check")
print(f"  Customers in transactions but not in customers.csv : {len(orphan_customers)}")
print(f"  Articles in transactions but not in articles.csv   : {len(orphan_articles)}")

if len(orphan_customers) == 0 and len(orphan_articles) == 0:
    print("\n  ✅ No orphan records found — relationships are intact!")
else:
    print("\n  ⚠️  Orphan records detected — will need investigation in preprocessing.")

---

## 📊 Section 8: Exploratory Data Analysis (EDA)

EDA is where we **visualize the data** to find patterns, trends, and anomalies.

We'll explore:
1. Top purchased products
2. Top active customers
3. Customer age distribution
4. Purchase volume over time
5. Transaction channel distribution

### 8.1 — Top 20 Most Purchased Articles

In [ ]:
# ──────────────────────────────────────────────
# Top 20 Most Purchased Articles
# ──────────────────────────────────────────────

# Count how many times each article was purchased
top_articles = (
    transactions['article_id']
    .value_counts()
    .head(20)
    .reset_index()
)
top_articles.columns = ['article_id', 'purchase_count']

# Join with article names for readable labels
top_articles = top_articles.merge(
    articles[['article_id', 'prod_name']],
    on='article_id',
    how='left'
)

# Truncate long product names for clean display
top_articles['label'] = top_articles['prod_name'].str[:35]

# ── Plot ──
fig, ax = plt.subplots(figsize=(13, 7))

bars = ax.barh(
    top_articles['label'][::-1],
    top_articles['purchase_count'][::-1],
    color=plt.cm.Blues(np.linspace(0.4, 0.85, 20))
)

ax.set_xlabel('Number of Purchases', fontsize=12)
ax.set_title('Top 20 Most Purchased Articles', fontsize=15, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Add value labels on bars
for bar in bars:
    width = bar.get_width()
    ax.text(width + 50, bar.get_y() + bar.get_height() / 2,
            f'{int(width):,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/raw/plot_top_articles.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💡 These are the most popular items — strong candidates for popularity-based baselines.")

### 8.2 — Top 20 Most Active Customers

In [ ]:
# ──────────────────────────────────────────────
# Top 20 Most Active Customers (by transaction count)
# ──────────────────────────────────────────────

top_customers = (
    transactions['customer_id']
    .value_counts()
    .head(20)
    .reset_index()
)
top_customers.columns = ['customer_id', 'purchase_count']

# Shorten customer_id for display
top_customers['short_id'] = top_customers['customer_id'].str[:12] + '...'

# ── Plot ──
fig, ax = plt.subplots(figsize=(13, 6))

bars = ax.bar(
    range(len(top_customers)),
    top_customers['purchase_count'],
    color=plt.cm.Oranges(np.linspace(0.4, 0.9, 20))
)

ax.set_xticks(range(len(top_customers)))
ax.set_xticklabels(top_customers['short_id'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Number of Purchases')
ax.set_title('Top 20 Most Active Customers', fontsize=15, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 1,
            f'{int(height):,}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../data/raw/plot_top_customers.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💡 High-activity customers provide rich interaction data — very valuable for training.")

### 8.3 — Customer Age Distribution

In [ ]:
# ──────────────────────────────────────────────
# Customer Age Distribution
# ──────────────────────────────────────────────

# Drop missing ages before plotting
age_data = customers['age'].dropna()

fig, ax = plt.subplots(figsize=(13, 5))

ax.hist(age_data, bins=50, color='steelblue', edgecolor='white', linewidth=0.5, alpha=0.85)

# Add median and mean lines
ax.axvline(age_data.median(), color='crimson',   linestyle='--', linewidth=1.8, label=f'Median: {age_data.median():.0f}')
ax.axvline(age_data.mean(),   color='darkorange', linestyle='--', linewidth=1.8, label=f'Mean:   {age_data.mean():.1f}')

ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Number of Customers', fontsize=12)
ax.set_title('Customer Age Distribution', fontsize=15, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../data/raw/plot_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Age Statistics:")
print(f"   Min: {age_data.min():.0f} | Max: {age_data.max():.0f} | Mean: {age_data.mean():.1f} | Median: {age_data.median():.0f}")
print(f"\n💡 Age can be a useful feature for recommendation — different age groups prefer different styles.")

### 8.4 — Purchase Volume Over Time

In [ ]:
# ──────────────────────────────────────────────
# Purchases Over Time (Weekly)
# ──────────────────────────────────────────────

# Group transactions by week to smooth daily noise
weekly_purchases = (
    transactions
    .set_index('t_dat')
    .resample('W')          # 'W' = weekly aggregation
    .size()
    .reset_index()
)
weekly_purchases.columns = ['week', 'purchase_count']

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(weekly_purchases['week'], weekly_purchases['purchase_count'],
                color='teal', alpha=0.3)
ax.plot(weekly_purchases['week'], weekly_purchases['purchase_count'],
        color='teal', linewidth=1.8)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Purchases per Week', fontsize=12)
ax.set_title('Weekly Purchase Volume Over Time', fontsize=15, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../data/raw/plot_purchases_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💡 Temporal trends (seasonality, peaks) are important context signals for recommendations.")

### 8.5 — Sales Channel Distribution

In [ ]:
# ──────────────────────────────────────────────
# Sales Channel Breakdown
# Channel 1 = Online | Channel 2 = In-store (typical)
# ──────────────────────────────────────────────

channel_counts = transactions['sales_channel_id'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
ax1.pie(
    channel_counts.values,
    labels=[f'Channel {c}' for c in channel_counts.index],
    autopct='%1.1f%%',
    colors=['#2196F3', '#FF9800'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax1.set_title('Sales Channel Distribution', fontsize=13, fontweight='bold')

# Bar chart
bars = ax2.bar(
    [f'Channel {c}' for c in channel_counts.index],
    channel_counts.values,
    color=['#2196F3', '#FF9800'],
    edgecolor='white'
)
ax2.set_ylabel('Transaction Count')
ax2.set_title('Sales Channel Count', fontsize=13, fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, height,
             f'{int(height):,}', ha='center', va='bottom')

plt.suptitle('Transaction Channel Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/raw/plot_sales_channels.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💡 Online vs. in-store behavior may differ significantly — channel can be a useful context feature.")

### 8.6 — Transaction Count Distribution per Customer

In [ ]:
# ──────────────────────────────────────────────
# How many purchases does each customer make?
# This reveals the 'long tail' problem in recommendation systems
# ──────────────────────────────────────────────

customer_txn_counts = transactions.groupby('customer_id').size()

fig, ax = plt.subplots(figsize=(13, 5))

# Cap at 100 purchases to avoid extreme outliers dominating the chart
capped = customer_txn_counts.clip(upper=100)
ax.hist(capped, bins=60, color='mediumpurple', edgecolor='white', linewidth=0.5, alpha=0.85)

ax.set_xlabel('Number of Purchases per Customer (capped at 100)', fontsize=12)
ax.set_ylabel('Number of Customers', fontsize=12)
ax.set_title('Purchase Frequency Distribution per Customer', fontsize=15, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('../data/raw/plot_customer_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Purchase Frequency Stats per Customer:")
print(customer_txn_counts.describe().to_string())
print(f"\n💡 Most customers have very few purchases — this is the 'cold start' and 'long tail' problem.")
print("   A small fraction of power users drive most transactions.")

---

## 💡 Section 9: Dataset Insights

### 9.1 — Business Insights

From our exploration, we can draw several meaningful business conclusions:

| Observation | Business Implication |
|-------------|----------------------|
| A small set of articles are purchased very frequently | Popularity-based recommendations are a strong baseline |
| Most customers have few purchases | Cold-start problem exists — we need strategies for new/inactive users |
| Purchases show temporal trends | Seasonality and recency matter in recommendations |
| Online vs. in-store channels differ | Channel-aware recommendations may improve relevance |
| Age distribution peaks in 20s–40s | Primary demographic is young-to-middle-aged adults |

---

### 9.2 — Recommendation Relevance Observations

- **Collaborative Filtering potential**: With millions of transactions, we have rich user-item interaction data — ideal for collaborative filtering approaches.
- **Content-Based potential**: The articles dataset contains rich metadata (product type, color, department) — useful for content-based or hybrid approaches.
- **Context signals**: Timestamps, sales channel, and customer age provide important context signals for a context-aware model.

---

### 9.3 — User Behavior Observations

- **Power users**: A small group of highly active customers make hundreds of purchases. These users provide the most training signal.
- **Infrequent users**: The majority of customers have only 1–5 purchases. Recommendations for these users will rely more on demographic and content signals.
- **Seasonal spikes**: Purchase peaks visible in the time series may correspond to sales events (Black Friday, seasonal collections).

In [ ]:
# ──────────────────────────────────────────────
# Summary Statistics: Quick Reference
# ──────────────────────────────────────────────

print("="*60)
print("  DATASET SUMMARY — Quick Reference")
print("="*60)

print(f"\n  👥 Total Customers                  : {len(customers):,}")
print(f"  👗 Total Articles (Products)         : {len(articles):,}")
print(f"  🧾 Total Transactions                : {len(transactions):,}")
print(f"  📅 Transaction Date Range            : {transactions['t_dat'].min().date()} → {transactions['t_dat'].max().date()}")

avg_price = transactions['price'].mean() if 'price' in transactions.columns else 'N/A'
if avg_price != 'N/A':
    print(f"  💰 Average Transaction Price         : {avg_price:.4f} (normalized)")

print(f"  🔗 Customers with purchases          : {transactions['customer_id'].nunique():,} ({transactions['customer_id'].nunique()/len(customers)*100:.1f}%)")
print(f"  🔗 Articles ever purchased           : {transactions['article_id'].nunique():,} ({transactions['article_id'].nunique()/len(articles)*100:.1f}%)")

avg_txn = transactions.groupby('customer_id').size().mean()
print(f"  📦 Avg transactions per customer     : {avg_txn:.1f}")

---

## ✅ Section 10: Initial Conclusions

### What We've Learned

This notebook gave us a thorough understanding of the H&M dataset. Here's a structured summary of findings and what comes next.

---

### 🔧 Preprocessing Steps Needed (Next Notebook)

| Issue Found | Action Needed |
|-------------|---------------|
| Missing `age` in customers | Impute with median or model age from behavior |
| Missing values in article metadata | Fill with 'Unknown' or drop sparse columns |
| Missing `club_member_status`, `fashion_news_frequency` | Encode as separate category |
| `article_id` must stay as string | Ensure consistent typing across all joins |
| Extreme outliers in purchase frequency | Consider capping or log-transforming for modeling |
| Timestamps need feature extraction | Extract week, month, season, day-of-week |

---

### 🎯 Features Likely Useful for Recommendation Modeling

**From Customers:**
- `age` — demographic signal
- `club_member_status` — engagement level
- `fashion_news_frequency` — interest signal

**From Articles:**
- `product_type_name`, `product_group_name` — category hierarchy
- `colour_group_name` — visual preference
- `department_name`, `section_name` — navigation context
- `garment_group_name` — item type

**From Transactions:**
- `t_dat` → derived: week, month, season, days_since_last_purchase
- `price` — price sensitivity signal
- `sales_channel_id` — online vs. in-store context

---

### 🗺️ What's Next

```
  Notebook 01: Data Understanding        ← YOU ARE HERE ✅
  Notebook 02: Data Preprocessing        ← Next step
  Notebook 03: Feature Engineering
  Notebook 04: Model Building (TF Recommenders)
  Notebook 05: Evaluation & Serving
```

> ✍️ **Note for the team**: All findings from this notebook should be referenced when making preprocessing and feature decisions in Notebook 02.

In [ ]:
# ──────────────────────────────────────────────
# End of Notebook
# ──────────────────────────────────────────────

print("🎉 Notebook 01: Master Data Understanding — COMPLETE")
print("\nSummary of datasets explored:")
print(f"  ✅ customers.csv         — {len(customers):,} rows")
print(f"  ✅ articles.csv          — {len(articles):,} rows")
print(f"  ✅ transactions_train.csv — {len(transactions):,} rows")
print("\nReady to proceed to Notebook 02: Data Preprocessing. 🚀")